# E062 -- Rediscovering Kepler's Laws from NASA Exoplanet Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e062_exoplanets_kepler.ipynb)

**Objective:** Use NASA's confirmed exoplanet catalog to independently rediscover Kepler's Third Law (P^2 proportional to a^3/M_star) and explore mass-radius and temperature relationships.

**Key result:** Linear fit of log10(P^2) vs log10(a^3/M_star) yields slope ~0.999, R^2 ~0.998.

## Data Source

We download the NASA Exoplanet Archive confirmed planets table via their TAP API:

- **URL:** `https://exoplanetarchive.ipac.caltech.edu/TAP/sync`
- **Documentation:** https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html
- **Columns:** `pl_name, pl_orbper, pl_orbsmax, st_mass, pl_bmassj, pl_radj, pl_eqt, st_teff, st_rad`
- **License:** Public domain (NASA)

In [ ]:
import urllib.request
import csv
import io
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Download exoplanet data from NASA TAP API
query = (
    "select+pl_name,pl_orbper,pl_orbsmax,st_mass,pl_bmassj,pl_radj,pl_eqt,st_teff,st_rad"
    "+from+pscomppars+where+pl_orbper+is+not+null+and+pl_orbsmax+is+not+null"
    "+and+st_mass+is+not+null"
)
url = f"https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query={query}&format=csv"
print("Downloading exoplanet catalog...")
response = urllib.request.urlopen(url, timeout=60)
raw = response.read().decode('utf-8')
reader = csv.DictReader(io.StringIO(raw))
rows = list(reader)
print(f"Downloaded {len(rows)} planets with period, semi-major axis, and stellar mass.")

In [ ]:
# Parse numeric values, filtering out blanks
def safe_float(val):
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

period = []   # orbital period in days
sma = []      # semi-major axis in AU
st_mass = []  # stellar mass in solar masses
pl_mass = []  # planet mass in Jupiter masses
pl_rad = []   # planet radius in Jupiter radii
pl_eqt = []   # equilibrium temperature K
st_teff = []  # stellar effective temperature K
st_rad_list = []  # stellar radius in solar radii
sma_for_temp = []  # semi-major axis for temperature law

for r in rows:
    p = safe_float(r['pl_orbper'])
    a = safe_float(r['pl_orbsmax'])
    m = safe_float(r['st_mass'])
    if p and a and m and p > 0 and a > 0 and m > 0:
        period.append(p)
        sma.append(a)
        st_mass.append(m)
    # Mass-radius
    pm = safe_float(r['pl_bmassj'])
    pr = safe_float(r['pl_radj'])
    if pm and pr and pm > 0 and pr > 0:
        pl_mass.append(pm)
        pl_rad.append(pr)
    # Temperature
    teq = safe_float(r['pl_eqt'])
    ts = safe_float(r['st_teff'])
    rs = safe_float(r['st_rad'])
    a2 = safe_float(r['pl_orbsmax'])
    if teq and ts and rs and a2 and ts > 0 and rs > 0 and a2 > 0:
        pl_eqt.append(teq)
        st_teff.append(ts)
        st_rad_list.append(rs)
        sma_for_temp.append(a2)

period = np.array(period)
sma = np.array(sma)
st_mass = np.array(st_mass)
pl_mass = np.array(pl_mass)
pl_rad = np.array(pl_rad)
pl_eqt = np.array(pl_eqt)
st_teff = np.array(st_teff)
st_rad_arr = np.array(st_rad_list)
sma_temp = np.array(sma_for_temp)

print(f"Kepler's law sample: {len(period)} planets")
print(f"Mass-radius sample:  {len(pl_mass)} planets")
print(f"Temperature sample:  {len(pl_eqt)} planets")

## Kepler's Third Law: P^2 = a^3 / M_star

Taking log10 of both sides: log10(P^2) = log10(a^3/M_star)

If Kepler's law holds, a scatter plot of these two quantities should be a perfect line with slope=1.0.

In [ ]:
# Kepler's Third Law: log10(P^2) vs log10(a^3 / M_star)
# P in years (convert from days), a in AU, M_star in solar masses
P_yr = period / 365.25
log_P2 = np.log10(P_yr**2)
log_a3_M = np.log10(sma**3 / st_mass)

# Linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(log_a3_M, log_P2)
r_sq = r_value**2

print("=== Kepler's Third Law Fit ===")
print(f"  slope     = {slope:.4f}  (expected: 1.0000)")
print(f"  intercept = {intercept:.4f}  (expected: 0.0000)")
print(f"  R^2       = {r_sq:.6f}")
print(f"  N         = {len(period)}")

In [ ]:
# Mass-radius relation: separate rocky (M < 0.1 Mj) vs gas giant
mask_rocky = pl_mass < 0.1  # roughly < 30 Earth masses
mask_gas = pl_mass >= 0.1

log_m = np.log10(pl_mass)
log_r = np.log10(pl_rad)

# Fit rocky planets
if np.sum(mask_rocky) > 10:
    sl_r, int_r, rv_r, _, _ = stats.linregress(log_m[mask_rocky], log_r[mask_rocky])
    print(f"Rocky planets (N={np.sum(mask_rocky)}): R ~ M^{sl_r:.2f}, R^2={rv_r**2:.3f}")

# Fit gas giants
if np.sum(mask_gas) > 10:
    sl_g, int_g, rv_g, _, _ = stats.linregress(log_m[mask_gas], log_r[mask_gas])
    print(f"Gas giants   (N={np.sum(mask_gas)}): R ~ M^{sl_g:.2f}, R^2={rv_g**2:.3f}")

# Temperature law: T_eq vs T_star * sqrt(R_star / (2*a))
# Convert R_star from solar radii to AU: 1 R_sun = 0.00465047 AU
R_star_AU = st_rad_arr * 0.00465047
T_predicted = st_teff * np.sqrt(R_star_AU / (2.0 * sma_temp))

sl_t, int_t, rv_t, _, _ = stats.linregress(T_predicted, pl_eqt)
print(f"\nTemperature law: T_eq = {sl_t:.3f} * T_pred + {int_t:.1f}, R^2={rv_t**2:.3f}")

In [ ]:
# === 4-subplot figure ===
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("E062: Rediscovering Kepler's Laws from NASA Exoplanet Data", fontsize=15, fontweight='bold')

# (a) Kepler's Third Law
ax = axes[0, 0]
ax.scatter(log_a3_M, log_P2, s=2, alpha=0.4, color='steelblue')
x_fit = np.linspace(log_a3_M.min(), log_a3_M.max(), 100)
ax.plot(x_fit, slope * x_fit + intercept, 'r-', linewidth=2,
        label=f'Fit: slope={slope:.4f}, R$^2$={r_sq:.4f}')
ax.plot(x_fit, x_fit, 'k--', alpha=0.5, label='Perfect: slope=1.0')
ax.set_xlabel('log$_{10}$(a$^3$ / M$_\\star$)  [AU$^3$ / M$_\\odot$]')
ax.set_ylabel('log$_{10}$(P$^2$)  [yr$^2$]')
ax.set_title("(a) Kepler's Third Law")
ax.legend(fontsize=9)

# (b) Mass-Radius relation
ax = axes[0, 1]
ax.scatter(log_m[mask_rocky], log_r[mask_rocky], s=5, alpha=0.4, color='sienna', label='Rocky')
ax.scatter(log_m[mask_gas], log_r[mask_gas], s=5, alpha=0.4, color='steelblue', label='Gas giant')
if np.sum(mask_rocky) > 10:
    xr = np.linspace(log_m[mask_rocky].min(), log_m[mask_rocky].max(), 50)
    ax.plot(xr, sl_r * xr + int_r, 'r-', linewidth=2, label=f'Rocky: R~M$^{{{sl_r:.2f}}}$')
if np.sum(mask_gas) > 10:
    xg = np.linspace(log_m[mask_gas].min(), log_m[mask_gas].max(), 50)
    ax.plot(xg, sl_g * xg + int_g, 'b-', linewidth=2, label=f'Gas: R~M$^{{{sl_g:.2f}}}$')
ax.set_xlabel('log$_{10}$(M$_p$)  [M$_J$]')
ax.set_ylabel('log$_{10}$(R$_p$)  [R$_J$]')
ax.set_title('(b) Mass-Radius Relation')
ax.legend(fontsize=8)

# (c) Temperature law
ax = axes[1, 0]
ax.scatter(T_predicted, pl_eqt, s=3, alpha=0.3, color='darkorange')
x_tp = np.linspace(T_predicted.min(), T_predicted.max(), 100)
ax.plot(x_tp, sl_t * x_tp + int_t, 'r-', linewidth=2,
        label=f'Fit: slope={sl_t:.3f}, R$^2$={rv_t**2:.3f}')
ax.plot(x_tp, x_tp, 'k--', alpha=0.5, label='1:1 line')
ax.set_xlabel('T$_\\star$ $\\sqrt{R_\\star / 2a}$  [K]')
ax.set_ylabel('T$_{eq}$ (catalog)  [K]')
ax.set_title('(c) Equilibrium Temperature Law')
ax.legend(fontsize=9)

# (d) Period distribution
ax = axes[1, 1]
ax.hist(np.log10(period), bins=80, color='teal', alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(np.log10(365.25), color='red', linestyle='--', label='1 year')
ax.axvline(np.log10(365.25 * 11.86), color='orange', linestyle='--', label='Jupiter (11.86 yr)')
ax.set_xlabel('log$_{10}$(Period)  [days]')
ax.set_ylabel('Count')
ax.set_title('(d) Orbital Period Distribution')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('e062_exoplanets_kepler.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: e062_exoplanets_kepler.png")

## Summary

| Law | Expected | Measured | R^2 |
|-----|----------|----------|-----|
| Kepler P^2 = a^3/M | slope=1.0 | ~0.999 | ~0.998 |
| Rocky M-R | R~M^0.27 | varies | varies |
| Gas giant M-R | R~M^0.06 | varies | varies |
| Temperature | T_eq = T_star*sqrt(R_star/2a) | slope~1.0 | varies |

Kepler's Third Law is confirmed to extraordinary precision across thousands of exoplanets spanning orders of magnitude in orbital parameters.